<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 通过 Ollama 使用 Llama 3 模型本地评估指令回复

- 本 notebook 通过 ollama 使用 80 亿参数的 Llama 3 模型，基于包含模型生成回复的 JSON 格式数据集评估指令微调 LLM 的回复，例如：



```python
{
    "instruction": "What is the atomic number of helium?",
    "input": "",
    "output": "The atomic number of helium is 2.",               # <-- The target given in the test set
    "model 1 response": "\nThe atomic number of helium is 2.0.", # <-- Response by an LLM
    "model 2 response": "\nThe atomic number of helium is 3."    # <-- Response by a 2nd LLM
},
```

- 代码无需 GPU，可在笔记本电脑上运行（已在 M3 MacBook Air 上测试）

In [ ]:
from importlib.metadata import version

pkgs = ["tqdm",    # 进度条
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

## 安装 Ollama 并下载 Llama 3

- Ollama 是一款高效运行 LLM 的应用
- 它是 [llama.cpp](https://github.com/ggerganov/llama.cpp) 的封装，后者用纯 C/C++ 实现 LLM 以最大化效率
- 注意，它是用于 LLM 文本生成（推理）的工具，而非训练或微调 LLM
- 运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按说明安装 ollama（例如点击「Download」按钮，下载适用于您操作系统的 ollama 应用）

- macOS 和 Windows 用户：点击您下载的 ollama 应用；若提示安装命令行工具，请选择「yes」
- Linux 用户：可使用 ollama 网站提供的安装命令

- 通常，要从命令行使用 ollama，需要先启动 ollama 应用，或在单独终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- 在 ollama 应用或 `ollama serve` 运行后，在另一个终端的命令行中执行以下命令试用 80 亿参数的 Llama 3 模型（该模型占用 4.7 GB 存储空间，首次执行此命令时会自动下载）

```bash
# 8B model
ollama run llama3
```


输出大致如下：

```
$ ollama run llama3
pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                          
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                          
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                          
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                          
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                          
verifying sha256 digest 
writing manifest 
removing any unused layers 
success 
```

- 注意，`llama3` 指指令微调版 80 亿参数的 Llama 3 模型

- 若您的机器支持，也可使用更大的 700 亿参数 Llama 3 模型，将 `llama3` 替换为 `llama3:70b` 即可

- 下载完成后，您将看到命令行提示符，可与模型对话

- 尝试输入 "What do llamas eat?" 等提示，应返回类似以下输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- 可使用 `/bye` 结束会话

## 使用 Ollama 的 REST API

- 现在，与模型交互的另一种方式是通过 Python 调用其 REST API，使用以下函数
- 运行本 notebook 后续单元格前，请确保 ollama 仍在运行，方式同上：
  - 在终端中运行 `ollama serve`
  - 或启动 ollama 应用
- 接下来，运行以下代码单元格查询模型

- 首先用简单示例测试 API，确保其按预期工作：

In [ ]:
import json
import requests


def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    # 将数据 payload 创建为字典
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {     # 以下设置用于确定性响应
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

result = query_model("What do Llamas eat?")
print(result)

## 加载 JSON 条目

- 现在进入数据评估部分
- 此处假设我们将测试集与模型回复保存为 JSON 文件，可按如下方式加载：

In [ ]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("条目数量:", len(json_data))

- 该文件结构如下，其中包含测试集中的给定回复（`'output'`）以及两个不同模型的回复（`'model 1 response'` 和 `'model 2 response'`）：

In [ ]:
json_data[0]

- 下面是一个小型工具函数，用于格式化输入，便于后续可视化：

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 现在，让我们尝试使用 ollama API 比较模型回复（我们仅评估前 5 条回复以便直观比较）：

In [ ]:
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\n数据集回复:")
    print(">>", entry['output'])
    print("\n模型回复:")
    print(">>", entry["model 1 response"])
    print("\n分数:")
    print(">>", query_model(prompt))
    print("\n-------------------------")

- 注意回复非常冗长；为量化哪个模型更好，我们只需返回分数：

In [ ]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key):
    scores = []
    for entry in tqdm(json_data, desc="评分条目"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue

    return scores

- 现在对整个数据集应用该评估并计算每个模型的平均分（在 M3 MacBook Air 笔记本电脑上每个模型大约需要 1 分钟）
- 注意，ollama 在不同操作系统上并非完全确定性（截至本文撰写时），因此您获得的数字可能与下面显示的略有不同

In [ ]:
from pathlib import Path

for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model)
    print(f"\n{model}")
    print(f"分数数量: {len(scores)} / {len(json_data)}")
    print(f"平均分数: {sum(scores)/len(scores):.2f}\n")

    # 可选：保存分数
    save_path = Path("scores") / f"llama3-8b-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)

- 根据上述评估，我们可以说第 1 个模型优于第 2 个模型